In [1]:
rm(list = ls())

# check and install CRAN packages
cran_packages <- c("ape", "ggtree", "ggplot2", "grid", "scales", "reshape2", "dplyr", "aplot")

for (pkg in cran_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
}

lapply(cran_packages, library, character.only = TRUE)


ggtree v3.16.3 Learn more at https://yulab-smu.top/contribution-tree-data/

Please cite:

Guangchuang Yu.  Data Integration, Manipulation and Visualization of
Phylogenetic Trees (1st edition). Chapman and Hall/CRC. 2022,
doi:10.1201/9781003279242, ISBN: 9781032233574


Attaching package: ‘ggtree’


The following object is masked from ‘package:ape’:

    rotate



Attaching package: ‘dplyr’


The following object is masked from ‘package:ape’:

    where


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


aplot v0.2.9 For help: https://github.com/YuLab-SMU/aplot/issues

If you use aplot in published research, please cite the paper:

Shuangbin Xu, Qianwen Wang, Shaodi Wen, Junrui Li, Nan He, Ming Li,
Thomas Hackl, Rui Wang, Dongqiang Zeng, Shixiang Wang, Shensuo Li,
Chunhui Gao, Lang Zhou, Shaoguo Tao, Zijing Xie, Lin Deng, and
Guangchuang Yu. aplot: Simplifying the cre

[[1]]
[1] "ape"       "repr"      "stats"     "graphics"  "grDevices" "utils"    
[7] "datasets"  "methods"   "base"     

[[2]]
 [1] "ggtree"    "ape"       "repr"      "stats"     "graphics"  "grDevices"
 [7] "utils"     "datasets"  "methods"   "base"     

[[3]]
 [1] "ggplot2"   "ggtree"    "ape"       "repr"      "stats"     "graphics" 
 [7] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[4]]
 [1] "grid"      "ggplot2"   "ggtree"    "ape"       "repr"      "stats"    
 [7] "graphics"  "grDevices" "utils"     "datasets"  "methods"   "base"     

[[5]]
 [1] "scales"    "grid"      "ggplot2"   "ggtree"    "ape"       "repr"     
 [7] "stats"     "graphics"  "grDevices" "utils"     "datasets"  "methods"  
[13] "base"     

[[6]]
 [1] "reshape2"  "scales"    "grid"      "ggplot2"   "ggtree"    "ape"      
 [7] "repr"      "stats"     "graphics"  "grDevices" "utils"     "datasets" 
[13] "methods"   "base"     

[[7]]
 [1] "dplyr"     "reshape2"  "scales"    "grid"      "ggplot2"   "ggtree"   
 [7] "ape"       "repr"      "stats"     "graphics"  "grDevices" "utils"    
[13] "datasets"  "methods"   "base"     

[[8]]
 [1] "aplot"     "dplyr"     "reshape2"  "scales"    "grid"      "ggplot2"  
 [7] "ggtree"    "ape"       "repr"      "stats"     "graphics"  "grDevices"
[13] "utils"     "datasets"  "methods"   "base"

In [2]:
treefile <- "./lib/OMAGroup_1072368.fa.trim.phy.contree"
tree <- read.tree(treefile)

spec_data <- read.delim2( "./lib/multi_species_list.txt", sep = "\t", header = TRUE)[1:55, ]
spec_data <- transform(
  spec_data,
  ID     = factor(ID),
  Family = factor(Family),
  Clade  = factor(Clade),
  Order  = factor(Order)
)

ppr_data <- read.csv("./lib/gma_ppr_class_sirna.csv", row.names = 1, check.names = FALSE)


In [3]:
p_tree <- ggtree(tree) +
  xlim_tree(xlim=c(0, 2)) + 
  geom_treescale(x=0, y=20, fontsize=2.5, linesize=0.4)

p_tree <- p_tree %<+% spec_data +
  geom_tiplab(
    size = 1.8, 
    align = TRUE, 
    offset = 0.01, # Distance from the tree tip
    linesize = 0.2, 
    linetype = "dotted"
  ) +
  guides(y = "none") +
  theme(
    legend.key.height = unit(0.4, 'cm'),
    legend.key.width = unit(0.4, 'cm'),
    legend.title = element_text(size=6, face = "bold"),
    legend.text = element_text(size=6)
  )

Warning message:
“`aes_()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`
ℹ The deprecated feature was likely used in the ggtree package.
  Please report the issue at <https://github.com/YuLab-SMU/ggtree/issues>.”
Warning message in fortify(data, ...):
“Arguments in `...` must be used.
✖ Problematic arguments:
• as.Date = as.Date
• yscale_mapping = yscale_mapping
• hang = hang
ℹ Did you misspell an argument name?”
Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.
ℹ The deprecated feature was likely used in the ggtree package.
  Please report the issue at <https://github.com/YuLab-SMU/ggtree/issues>.”
Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.
ℹ The deprecated feature was likely used in the ggtree package.
  Please report the issue at <https://gith

In [4]:
prep_long <- function(df, cols, value_name) {
  out <- df[, cols]
  names(out) <- sub("_.+", "", names(out))
  out <- data.frame(lapply(out, as.numeric), check.names = FALSE)
  out$ID <- rownames(df)
  melt(out, id.vars = "ID", variable.name = "Class", value.name = value_name)
}

d2_long <- prep_long(ppr_data, 13:17, "Number") |>
  filter(Class != "E+") |>
  mutate(
    ID = factor(ID, levels = rev(tree$tip.label)),
    Class = factor(Class, levels = c("DYW", "E", "P", "PLS"))
  )


In [5]:
breaks  <- c(-Inf, 0, 5, 15, 30, 45, max(d2_long$Number, na.rm = TRUE) + 1)
labels  <- c("0", "1-5", "6-15", "16-30", "31-45", "46-61")

d2_long$Category <- cut(
  d2_long$Number,
  breaks = breaks,
  labels = labels,
  include.lowest = TRUE
)

p_heat <- ggplot(d2_long, aes(Class, ID, fill = Category)) +
  geom_tile(color = "black", size = 0.25) +
  scale_x_discrete(position = "top", expand = c(0, 0)) +
  scale_y_discrete(expand = c(0, 0)) +
  scale_fill_manual(
    values = c(
      "0" = "white",
      "1-5" = "#EDA59C",
      "6-15" = "#DC5D5B",
      "16-30" = "#B72C32",
      "31-45" = "#8D1E26",
      "46-61" = "#5F0F1D"
    ),
    name = "Number",
    na.value = "grey90"
  ) +
  labs(x = "", y = "") +
  theme_minimal() +
  theme(
    axis.text.x = element_text(size = 6, angle = 90),
    axis.text.y = element_blank(),
    axis.ticks.y = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, size = 0.5),
    panel.grid = element_blank(),
    legend.title = element_text(size = 6, face = "bold"),
    legend.text = element_text(size = 6)
  )


Warning message:
“The `size` argument of `element_rect()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.”


In [6]:
# ass PPR-siRNA abundance

d3_long <- prep_long(ppr_data, 7:11, "TPM") |>
  filter(Class != "E+") |>
  mutate(
    ID = factor(ID, levels = rev(tree$tip.label)),
    Class = factor(Class, levels = rev(c("P", "DYW", "E", "PLS"))),
    TPM_log2 = log2(TPM + 1)
  )

p_bar <- ggplot(d3_long, aes(ID, TPM_log2, fill = Class)) +
  geom_col(width = 0.8, color = "black", size = 0.25) +
  coord_flip(clip = "off") +
  scale_y_continuous(
    limits = c(0, 20),
    expand = c(0, 0),
    name = expression(bold(log[2](TPM + 1)))
  ) +
  scale_fill_manual(
    values = alpha(
      c(DYW = "#e6194B", PLS = "#f58231", P = "#4363d8", E = "#ffe119"),
      0.8
    ),
    name = "Class"
  ) +
  labs(x = "", y = "") +
  theme_minimal() +
  theme(
    axis.text.y = element_blank(),
    axis.text.x = element_text(size = 6),
    panel.border = element_rect(color = "black", fill = NA, size = 0.25),
    panel.grid = element_blank(),
    legend.title = element_text(size = 6, face = "bold"),
    legend.text = element_text(size = 6)
  )


Warning message in geom_col(width = 0.8, color = "black", size = 0.25):
“Ignoring unknown parameters: `size`”


In [7]:
p <- p_heat %>%
  insert_left(p_tree, width = 5) %>%
  insert_right(p_bar, width = 1.5)

ggsave(
  filename = "ggtree_multispecies_PPR.pdf",
  plot = p,
  width = 10,
  height = 8
)
